[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab4_Deinterleaving_multiemisor.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 4 · Escenario multi-emisor: deinterleaving
### Separar los pulsos de varios radares que llegan intercalados al receptor

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 3 a 4 horas |
| 📥 **Datos de entrada** | `D3_ESM_escenario_multiemisor.xlsx` |
| 📤 **Producto al final** | Lista de emisores detectados (Excel) + línea de tiempo |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Explicar el problema del **deinterleaving** y por qué la PRI no se puede medir sobre la mezcla.
2. Explorar el espacio de parámetros (RF, AOA, PW) con histogramas, dispersiones y *pairplot*.
3. Segmentar por **banda** y definir **ventanas** (*gates*) de RF/AOA/PW para separar emisores.
4. Caracterizar cada grupo con las herramientas del Lab 3 y construir la **lista de emisores**.
5. Construir la **línea de tiempo de actividad** y detectar la aparición de nuevos emisores.
6. Reconocer los pulsos de ruido y las limitaciones del método manual.

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

In [ ]:
# @title 🧰 Caja de herramientas (Labs 1–4) — ejecutar
# ------------------------------------------------------------------
# A. CARGA Y LIMPIEZA DE PDW (lo que aprendiste a mano en el Lab 1)
# ------------------------------------------------------------------
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4)

CARPETAS_DATOS = [".", "datos", "/content", "/content/datos", "/content/sample_data",
                  "/content/drive/MyDrive", "/content/drive/MyDrive/datos",
                  "/content/drive/MyDrive/Analisis_de_Señales"]


def buscar_archivo(nombre):
    """Busca 'nombre' en las carpetas habituales (local y Google Colab)."""
    for carpeta in CARPETAS_DATOS:
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(
        f"No encontré '{nombre}'. Súbelo con el ícono de carpeta 📁 (panel izquierdo de Colab) "
        f"o copia su ruta exacta en la variable del archivo.")


def estandarizar_columnas(df):
    """Renombra las columnas de la exportación ESM a nombres estándar con unidades."""
    nuevos = {}
    for col in df.columns:
        c = str(col).upper()
        if c.startswith("UNNAMED") or c.strip() == "":
            continue
        if "TOA" in c:
            nuevos[col] = "TOA_us"
        elif "FREQ" in c or "FREC" in c:
            nuevos[col] = "FREQ_MHz"
        elif c.startswith("PW") or "WIDTH" in c:
            nuevos[col] = "PW_us"
        elif "AMP" in c:
            nuevos[col] = "AMP_dBm"
        elif "AOA" in c or "DOA" in c or "BEARING" in c:
            nuevos[col] = "AOA_deg"
        elif "PRI" in c:
            nuevos[col] = "PRI_us"
    columnas_vacias = [c for c in df.columns if c not in nuevos]
    return df.drop(columns=columnas_vacias).rename(columns=nuevos)


def limpiar_pdw(df, verbose=True):
    """Limpieza estándar de PDW: centinelas, vacíos, duplicados y orden por TOA."""
    n0 = len(df)
    df = df.copy()
    claves = [c for c in ["TOA_us", "FREQ_MHz", "PW_us"] if c in df.columns]
    df = df.dropna(subset=claves)
    n1 = len(df)
    malos = (df["FREQ_MHz"] <= 0) | (df["PW_us"] <= 0)      # registros "fantasma" del receptor
    df = df[~malos]
    n2 = len(df)
    df = df.drop_duplicates()
    n3 = len(df)
    df = df.sort_values("TOA_us").reset_index(drop=True)
    df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
    df["DTOA_us"] = df["TOA_us"].diff()
    if verbose:
        print(f"Filas originales ............ {n0:>7,}")
        print(f"  − con celdas vacías ....... {n0 - n1:>7,}")
        print(f"  − con ceros / centinela ... {n1 - n2:>7,}")
        print(f"  − duplicadas .............. {n2 - n3:>7,}")
        print(f"Filas válidas ............... {n3:>7,}")
    return df


# Carpeta web con los datos del curso (repositorio público). Déjala vacía ("") para trabajar sólo con archivos locales.
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"


def asegurar_archivo(nombre):
    """Devuelve la ruta del archivo; si no está, lo descarga (URL_DATOS) o pide subirlo (Colab)."""
    try:
        return buscar_archivo(nombre)
    except FileNotFoundError:
        pass
    if URL_DATOS:
        import urllib.request, urllib.parse
        os.makedirs("datos", exist_ok=True)
        destino = os.path.join("datos", nombre)
        try:
            urllib.request.urlretrieve(URL_DATOS.rstrip("/") + "/" + urllib.parse.quote(nombre), destino)
            print(f"⬇️ '{nombre}' descargado desde el repositorio del curso")
            return destino
        except Exception as e:
            print(f"⚠️ No se pudo descargar '{nombre}' ({e}). Intentando otras opciones…")
            if os.path.exists(destino):
                os.remove(destino)
    try:
        from google.colab import files
        print(f"📁 No encuentro '{nombre}'. Selecciónalo desde tu computador:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
    except ImportError:
        pass
    return buscar_archivo(nombre)   # mensaje de error claro


def cargar_pdw(nombre_archivo, hoja=0, verbose=True):
    """Carga un archivo ESM (Excel o CSV), estandariza columnas y limpia."""
    ruta = asegurar_archivo(nombre_archivo)
    if ruta.lower().endswith(".csv"):
        crudo = pd.read_csv(ruta)
    else:
        crudo = pd.read_excel(ruta, sheet_name=hoja)
    df = limpiar_pdw(estandarizar_columnas(crudo), verbose=verbose)
    if verbose:
        dur = df["TOA_s"].iloc[-1]
        print(f"Duración de la captura ...... {dur:,.2f} s")
    return df
# ------------------------------------------------------------------
# B. ESTADÍSTICA DESCRIPTIVA (Lab 2)
# ------------------------------------------------------------------
def moda_continua(serie, bins="auto"):
    """Moda de una variable continua = centro de la barra más alta del histograma."""
    x = pd.Series(serie).dropna().values
    cuentas, bordes = np.histogram(x, bins=bins)
    i = np.argmax(cuentas)
    return 0.5 * (bordes[i] + bordes[i + 1])


def ic95_media(serie, confianza=0.95):
    """Intervalo de confianza para la MEDIA usando la distribución t de Student."""
    x = pd.Series(serie).dropna().values
    n = len(x)
    media = x.mean()
    ee = x.std(ddof=1) / np.sqrt(n)              # error estándar de la media
    t = stats.t.ppf(0.5 + confianza / 2, df=n - 1)
    return media - t * ee, media + t * ee


def ic95_mediana_bootstrap(serie, n_boot=2000, confianza=0.95, semilla=0):
    """Intervalo de confianza para la MEDIANA por remuestreo (bootstrap)."""
    x = pd.Series(serie).dropna().values
    rng = np.random.default_rng(semilla)
    medianas = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    a = (1 - confianza) / 2
    return np.quantile(medianas, a), np.quantile(medianas, 1 - a)


def resumen_estadistico(serie, nombre=None):
    """Tabla con los estadígrafos más usados en un informe de análisis."""
    x = pd.Series(serie).dropna()
    lo, hi = ic95_media(x)
    return pd.Series({
        "n": int(len(x)),
        "media": x.mean(),
        "IC95_media_inf": lo,
        "IC95_media_sup": hi,
        "mediana": x.median(),
        "moda (histograma)": moda_continua(x),
        "desv_estandar": x.std(ddof=1),
        "CV_%": 100 * x.std(ddof=1) / abs(x.mean()) if x.mean() != 0 else np.nan,
        "minimo": x.min(),
        "P5": x.quantile(0.05),
        "Q1 (P25)": x.quantile(0.25),
        "Q3 (P75)": x.quantile(0.75),
        "P95": x.quantile(0.95),
        "maximo": x.max(),
        "rango": x.max() - x.min(),
        "IQR": x.quantile(0.75) - x.quantile(0.25),
        "asimetria": stats.skew(x),
    }, name=nombre or getattr(serie, "name", "variable"))


def graficar_distribucion(serie, titulo="", unidad="", bins=60):
    """Histograma + KDE, boxplot y violín de una misma variable (3 vistas)."""
    x = pd.Series(serie).dropna()
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2.2, 1, 1]})
    sns.histplot(x, bins=bins, kde=True, ax=ax[0], color="#4C72B0")
    ax[0].axvline(x.mean(), color="#C44E52", ls="--", lw=1.5, label=f"media = {x.mean():.4g}")
    ax[0].axvline(x.median(), color="#55A868", ls="-", lw=1.5, label=f"mediana = {x.median():.4g}")
    ax[0].legend(fontsize=9)
    ax[0].set_title(f"Histograma + KDE · {titulo}")
    ax[0].set_xlabel(unidad)
    sns.boxplot(y=x, ax=ax[1], color="#8172B2", width=0.4)
    ax[1].set_title("Boxplot"); ax[1].set_ylabel(unidad)
    sns.violinplot(y=x, ax=ax[2], color="#64B5CD", inner="quartile", cut=0)
    ax[2].set_title("Violín"); ax[2].set_ylabel(unidad)
    plt.tight_layout()
    plt.show()
# ------------------------------------------------------------------
# C. CARACTERIZACIÓN DE UN EMISOR (Lab 3)
# ------------------------------------------------------------------
from scipy.signal import find_peaks
from collections import Counter

C_LUZ = 299_792_458.0  # m/s


def banda_ieee(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.3, "VHF"), (1, "UHF"), (2, "L"), (4, "S"), (8, "C"), (12, "X"),
                   (18, "Ku"), (27, "K"), (40, "Ka"), (75, "V"), (110, "W")]:
        if f < lim:
            return b
    return "mm"


def banda_otan(f_mhz):
    f = f_mhz / 1000.0
    for lim, b in [(0.25, "A"), (0.5, "B"), (1, "C"), (2, "D"), (3, "E"), (4, "F"), (6, "G"),
                   (8, "H"), (10, "I"), (20, "J"), (40, "K"), (60, "L"), (100, "M")]:
        if f < lim:
            return b
    return ">M"


def analizar_pri(toa_us, verbose=True):
    """Analiza la secuencia de llegadas de UN emisor y sugiere el tipo de PRI.

    Pasos: (1) DTOA = diferencias de TOA; (2) PRI base = valor más bajo frecuente;
    (3) se descartan huecos de iluminación y pulsos perdidos (DTOA > 1.8·PRI base);
    (4) se buscan niveles discretos y se mide la correlación pulso a pulso."""
    toa = np.sort(np.asarray(toa_us, float))
    d = np.diff(toa)
    d = d[d > 0]
    r = {"n_dtoa": len(d)}
    if len(d) < 10:
        r["tipo"] = "Insuficientes pulsos"
        return r
    # (2) PRI base sobre escala logarítmica (bins de 0.5 %)
    ld = np.log(d)
    lb = np.arange(ld.min(), np.quantile(ld, 0.98) + 0.01, 0.005)
    if len(lb) < 3:
        lb = np.linspace(ld.min(), ld.max() + 1e-6, 5)
    cnt, _ = np.histogram(ld, bins=lb)
    acum = np.convolve(cnt, np.ones(7), mode="same")
    i0 = int(np.argmax(acum >= 0.03 * len(d)))
    base = float(np.exp(lb[i0]))
    principal = (d >= 0.8 * base) & (d <= 1.8 * base)
    m = d[principal]
    r.update(pri_base=base, fraccion_principal=principal.mean(),
             fraccion_perdidos=(d > 1.8 * base).mean())
    med = np.median(m)
    r.update(pri_mediana=med, pri_media=m.mean(), pri_min=np.quantile(m, 0.01),
             pri_max=np.quantile(m, 0.99), pri_std=m.std(ddof=1),
             pri_cv_pct=100 * m.std(ddof=1) / m.mean())
    # (4a) niveles discretos
    ancho = max(0.003 * med, 0.05)
    bordes = np.arange(m.min() - ancho, m.max() + 2 * ancho, ancho)
    h, _ = np.histogram(m, bins=bordes)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    picos, _ = find_peaks(np.r_[0, h, 0], height=0.08 * h.max(), distance=3)
    picos = picos - 1
    niveles = []
    for p in picos:
        sel = np.abs(m - centros[p]) <= 1.5 * ancho
        if sel.sum():
            niveles.append(float(np.median(m[sel])))
    niveles = sorted(niveles)
    conc = np.mean(np.min(np.abs(m[:, None] - np.array(niveles)[None, :]), axis=1) <= 2 * ancho) if niveles else 0
    discreto = (1 <= len(niveles) <= 8) and conc >= 0.85
    # (4b) correlación entre PRI consecutivos (sólo pares sin pulsos perdidos)
    ok = principal[:-1] & principal[1:]
    a, b = d[:-1][ok], d[1:][ok]
    r1 = float(np.corrcoef(a, b)[0, 1]) if len(a) > 5 and a.std() > 0 and b.std() > 0 else 0.0
    r.update(niveles=niveles, concentracion=conc, autocorr_lag1=r1)
    # (5) clasificación
    if r["fraccion_principal"] < 0.4:
        r["tipo"] = "Sin estructura (posible ruido o mezcla)"
        r["detalle"] = (f"sólo {100 * r['fraccion_principal']:.0f} % de los DTOA forman un patrón: "
                        "no hay un tren de pulsos coherente")
        if verbose:
            print(f"Tipo de PRI sugerido : {r['tipo']}\nDetalle              : {r['detalle']}")
        return r
    if discreto and len(niveles) == 1:
        tipo = "Estable"
        detalle = f"PRI ≈ {niveles[0]:.2f} µs (PRF ≈ {1e6 / niveles[0]:.1f} Hz), CV = {r['pri_cv_pct']:.3f} %"
    elif discreto:
        niv = np.array(niveles)
        idx = np.argmin(np.abs(m[:, None] - niv[None, :]), axis=1)
        corridas = np.diff(np.flatnonzero(np.r_[True, idx[1:] != idx[:-1], True]))
        largo_corrida = corridas.mean()
        if largo_corrida >= 4:
            tipo = "Dwell & Switch"
            detalle = (f"{len(niv)} PRI: " + ", ".join(f"{v:.1f}" for v in niv) +
                       f" µs · ~{largo_corrida:.0f} pulsos por permanencia (dwell)")
            r["pulsos_por_dwell"] = largo_corrida
        else:
            tipo = f"Stagger {len(niv)} posiciones"
            k = len(niv)
            ventanas = [tuple(idx[i:i + k]) for i in range(len(idx) - k)]
            ventanas = [w for w in ventanas if len(set(w)) == k]
            seq = Counter(ventanas).most_common(1)[0][0] if ventanas else tuple(range(k))
            orden = [niv[j] for j in seq]
            r["secuencia"] = orden
            r["periodo_trama"] = float(niv.sum())
            detalle = ("secuencia " + " → ".join(f"{v:.1f}" for v in orden) +
                       f" µs · periodo de trama {niv.sum():.1f} µs · PRI media {niv.mean():.1f} µs")
    else:
        if r1 > 0.6:
            tipo = "Deslizante (sliding)"
            caidas = np.flatnonzero(np.diff(m) < -0.5 * (r["pri_max"] - r["pri_min"]))
            pulsos_rampa = float(np.median(np.diff(caidas))) if len(caidas) > 2 else np.nan
            r["pulsos_por_rampa"] = pulsos_rampa
            detalle = (f"rampa de {r['pri_min']:.0f} a {r['pri_max']:.0f} µs, ~{pulsos_rampa:.0f} pulsos por rampa")
        else:
            tipo = "Jitter (aleatorio)"
            pct = 100 * (r["pri_max"] - r["pri_min"]) / (2 * med)
            r["jitter_pct"] = pct
            detalle = f"PRI central ≈ {med:.1f} µs, variación ≈ ±{pct:.1f} %"
    r["tipo"] = tipo
    r["detalle"] = detalle
    if verbose:
        print(f"Tipo de PRI sugerido : {tipo}")
        print(f"Detalle              : {detalle}")
        print(f"Pulsos perdidos      : {100 * r['fraccion_perdidos']:.1f} % de los DTOA (múltiplos de la PRI o huecos de scan)")
    return r


def graficar_pri(toa_us, titulo="", max_pulsos=600):
    """Dos vistas clásicas: secuencia DTOA vs nº de pulso, e histograma de DTOA."""
    d = np.diff(np.sort(np.asarray(toa_us, float)))
    lim = np.quantile(d, 0.97) * 1.3
    fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
    n = min(len(d), max_pulsos)
    ax[0].plot(np.arange(n), d[:n], ".", ms=4)
    ax[0].set_ylim(0, lim)
    ax[0].set_xlabel("Nº de pulso"); ax[0].set_ylabel("DTOA (µs)")
    ax[0].set_title(f"Secuencia de PRI · {titulo}")
    ax[1].hist(d[d < lim], bins=200, color="#4C72B0")
    ax[1].set_xlabel("DTOA (µs)"); ax[1].set_ylabel("Nº de pulsos")
    ax[1].set_title("Histograma de DTOA")
    plt.tight_layout(); plt.show()


def detectar_iluminaciones(df, gap_us=100_000, min_pulsos=5):
    """Agrupa pulsos en 'iluminaciones' (ráfagas) separadas por huecos > gap_us.
    Se ignoran grupos con menos de 'min_pulsos' (pulsos sueltos de lóbulos laterales)."""
    toa = df["TOA_us"].values
    tsec = df["TOA_s"].values if "TOA_s" in df else toa / 1e6     # segundos desde el inicio de la captura
    amp = df["AMP_dBm"].values
    corte = np.r_[0, np.flatnonzero(np.diff(toa) > gap_us) + 1, len(toa)]
    filas = []
    for i, j in zip(corte[:-1], corte[1:]):
        if j - i < min_pulsos:
            continue
        t, a = toa[i:j], amp[i:j]
        sel = a >= a.max() - 3.0                       # pulsos dentro de 3 dB del máximo
        w = 10 ** (a[sel] / 10)
        # ancho del lóbulo principal: ajuste de una parábola (en dB) a los pulsos
        # que están a menos de 10 dB del máximo  →  ancho a -3 dB = 2·sqrt(3/|c2|)
        lob = a >= a.max() - 10.0
        tot = np.nan
        if lob.sum() >= 5:
            tc = (t[lob] - t[lob].mean()) / 1e3                      # ms
            c2 = np.polyfit(tc, a[lob], 2)[0]
            if c2 < 0:
                tot = 2 * np.sqrt(3.0 / -c2)
        ts = tsec[i:j]
        filas.append({
            "t_inicio_s": ts[0], "t_fin_s": ts[-1], "n_pulsos": j - i,
            "duracion_ms": (t[-1] - t[0]) / 1e3,
            "t_centro_s": np.sum(ts[sel] * w) / np.sum(w),
            "amp_max_dBm": a.max(),
            "tot_3dB_ms": tot,
        })
    return pd.DataFrame(filas)


def espectro_amplitud(df, fmax=200.0):
    """Espectro (FFT) de la amplitud: revela modulaciones periódicas (scan cónico)."""
    t = df["TOA_us"].values / 1e6
    a = df["AMP_dBm"].values
    fs = min(1.0 / np.median(np.diff(t)), 4 * fmax)
    tu = np.arange(t[0], t[-1], 1 / fs)
    au = np.interp(tu, t, a)
    au = (au - au.mean()) * np.hanning(len(au))
    esp = np.abs(np.fft.rfft(au)) / len(au)
    f = np.fft.rfftfreq(len(au), 1 / fs)
    sel = (f >= 2) & (f <= fmax)
    return f[sel], esp[sel]


def analizar_scan(df, gap_us=100_000, verbose=True):
    """Clasifica el tipo de exploración de antena a partir de AMP vs TOA."""
    il = detectar_iluminaciones(df, gap_us)
    r = {"n_iluminaciones": len(il)}
    durac_total = (df["TOA_us"].iloc[-1] - df["TOA_us"].iloc[0]) / 1e6
    if len(il) == 0:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="pulsos dispersos, sin iluminaciones definidas")
        if verbose:
            print(f"Tipo de scan sugerido : {r['tipo']}\nDetalle               : {r['detalle']}")
        return r
    if len(il) <= 2 and il["duracion_ms"].max() / 1e3 > 0.6 * durac_total:
        f, e = espectro_amplitud(df)
        k = int(np.argmax(e))
        prominencia = e[k] / (np.median(e) + 1e-12)
        prof = df["AMP_dBm"].quantile(0.95) - df["AMP_dBm"].quantile(0.05)
        if prominencia > 8 and prof > 1.5:
            r.update(tipo="Cónico", frecuencia_scan_Hz=float(f[k]), periodo_s=1 / float(f[k]),
                     profundidad_dB=float(prof),
                     detalle=f"modulación de amplitud a {f[k]:.1f} Hz (periodo {1000 / f[k]:.1f} ms), profundidad ≈ {prof:.1f} dB")
        else:
            r.update(tipo="Fijo (iluminación continua)", periodo_s=0.0,
                     detalle=f"amplitud estable (variación {prof:.1f} dB): posible SEGUIMIENTO / lock-on")
    elif len(il) == 2:
        dt2 = float(il["t_centro_s"].iloc[1] - il["t_centro_s"].iloc[0])
        r.update(tipo="Indeterminado (2 iluminaciones)", periodo_s=dt2,
                 detalle=f"sólo 2 iluminaciones separadas {dt2:.2f} s: periodo probable, BAJA confianza (captura más larga)")
    elif len(il) < 3:
        r.update(tipo="Indeterminado", periodo_s=np.nan, detalle="una sola iluminación: no se puede estimar el periodo")
    else:
        dt = np.diff(il["t_centro_s"].values)
        cv = dt.std() / dt.mean()
        if cv < 0.08:
            T = float(np.median(dt))
            tot = float(il["tot_3dB_ms"].median())
            haz = 360.0 * tot / 1000 / T
            r.update(tipo="Circular", periodo_s=T, rpm=60 / T, tot_3dB_ms=tot, haz_estimado_deg=haz,
                     detalle=f"periodo {T:.2f} s ({60 / T:.1f} rpm) · tiempo en blanco (3 dB) {tot:.1f} ms · haz ≈ {haz:.1f}°")
        else:
            pares = dt[:-1] + dt[1:]
            alterna = np.mean(np.sign(np.diff(dt)[:-1]) != np.sign(np.diff(dt)[1:])) > 0.7
            if alterna and pares.std() / pares.mean() < 0.08:
                T = float(np.median(pares))
                r.update(tipo="Sectorial (bidireccional)", periodo_s=T,
                         detalle=f"iluminaciones en pares alternos ({np.median(dt[::2]):.2f} s / {np.median(dt[1::2]):.2f} s) · periodo de barrido {T:.2f} s")
            else:
                r.update(tipo="Irregular", periodo_s=float(np.median(dt)),
                         detalle="intervalos irregulares: posible cambio de modo, exploración electrónica o varios emisores mezclados")
        largas = il["duracion_ms"] > 5 * il["duracion_ms"].median()
        if largas.any():
            r["detalle"] += f" · ⚠ {int(largas.sum())} iluminación(es) prolongada(s): posible paso a SEGUIMIENTO"
    if verbose:
        print(f"Tipo de scan sugerido : {r['tipo']}")
        print(f"Detalle               : {r['detalle']}")
    return r


def media_circular(angulos_deg):
    """Media de ángulos respetando el salto 359° → 0°."""
    a = np.deg2rad(np.asarray(angulos_deg, float))
    return float(np.rad2deg(np.arctan2(np.sin(a).mean(), np.cos(a).mean())) % 360)


def ficha_emisor(df, nombre="EMISOR", gap_us=100_000, verbose=True):
    """Ficha paramétrica completa de un emisor (RF, PW, PRI, scan, AOA)."""
    f = df["FREQ_MHz"]
    rango_rf = f.quantile(0.99) - f.quantile(0.01)
    agil = rango_rf > 15
    pri = analizar_pri(df["TOA_us"], verbose=False)
    scan = analizar_scan(df, gap_us=gap_us, verbose=False)
    ficha = {
        "EMISOR": nombre,
        "N_PULSOS": len(df),
        "T_INICIO_s": df["TOA_s"].min() if "TOA_s" in df else np.nan,
        "T_FIN_s": df["TOA_s"].max() if "TOA_s" in df else np.nan,
        "RF_MEDIANA_MHz": round(f.median(), 1),
        "RF_MIN_MHz": round(f.quantile(0.01), 1),
        "RF_MAX_MHz": round(f.quantile(0.99), 1),
        "RF_TIPO": "Ágil" if agil else "Fija",
        "BANDA_IEEE": banda_ieee(f.median()),
        "BANDA_OTAN": banda_otan(f.median()),
        "PW_MEDIANA_us": round(df["PW_us"].median(), 2),
        "PW_P5_us": round(df["PW_us"].quantile(0.05), 2),
        "PW_P95_us": round(df["PW_us"].quantile(0.95), 2),
        "PRI_TIPO": pri.get("tipo"),
        "PRI_DETALLE": pri.get("detalle"),
        "PRI_MIN_us": round(pri.get("pri_min", np.nan), 1),
        "PRI_MAX_us": round(pri.get("pri_max", np.nan), 1),
        "PRI_MEDIA_us": round(pri.get("pri_media", np.nan), 1),
        "PRF_MEDIA_Hz": round(1e6 / pri.get("pri_media", np.nan), 1),
        "SCAN_TIPO": scan.get("tipo"),
        "SCAN_PERIODO_s": round(scan.get("periodo_s", np.nan), 3),
        "SCAN_DETALLE": scan.get("detalle"),
        "AOA_MEDIA_deg": round(media_circular(df["AOA_deg"]), 1) if "AOA_deg" in df else np.nan,
        "AMP_MAX_dBm": round(df["AMP_dBm"].max(), 1),
    }
    if verbose:
        for k, v in ficha.items():
            print(f"{k:<16}: {v}")
    return ficha
# ------------------------------------------------------------------
# D. DEINTERLEAVING (Lab 4)
# ------------------------------------------------------------------
def asignar_por_ventanas(df, ventanas, etiqueta="GRUPO"):
    """Asigna cada pulso al primer grupo cuyas ventanas (mín, máx) contengan sus parámetros."""
    df = df.copy()
    df[etiqueta] = "SIN_ASIGNAR"
    for grupo, limites in ventanas.items():
        sel = df[etiqueta] == "SIN_ASIGNAR"
        for col, (lo, hi) in limites.items():
            sel &= df[col].between(lo, hi)
        df.loc[sel, etiqueta] = grupo
    return df


def histograma_diferencias(toa_us, orden_max=4, max_us=3000, bins=600, titulo="", log=False):
    """Histograma de diferencias de TOA de orden 1..k (idea base de CDIF/SDIF)."""
    toa = np.sort(np.asarray(toa_us, float))
    fig, ax = plt.subplots(orden_max, 1, figsize=(13, 2.0 * orden_max), sharex=True)
    bordes = np.linspace(0, max_us, bins + 1)
    centros = 0.5 * (bordes[1:] + bordes[:-1])
    for k in range(1, orden_max + 1):
        d = toa[k:] - toa[:-k]
        h, _ = np.histogram(d, bins=bordes)
        ax[k - 1].fill_between(centros, h + (1 if log else 0), step="mid", color="#4C72B0")
        if log:
            ax[k - 1].set_yscale("log")
        ax[k - 1].set_ylabel(f"orden {k}")
    ax[0].set_title(f"Histogramas de diferencias de TOA · {titulo}")
    ax[-1].set_xlabel("Diferencia de TOA (µs)")
    plt.tight_layout(); plt.show()
print('Caja de herramientas lista ✔')

---
## 1. El problema: todos los radares "hablan" a la vez

📘 Un receptor ESM recibe **simultáneamente** los pulsos de todos los radares cuyo haz lo alcanza. Lo que registra es un único tren de pulsos **intercalados** (*interleaved*). Separarlos por emisor se llama **deinterleaving** y es la tarea central del procesamiento ESM/ELINT.

In [ ]:
rng = np.random.default_rng(0)
trenes = {"Radar A (PRI 100)": np.arange(3, 600, 100.0),
          "Radar B (PRI 137)": np.arange(40, 600, 137.0),
          "Radar C (jitter ~90)": np.cumsum(rng.uniform(70, 110, 8)) }
fig, ax = plt.subplots(4, 1, figsize=(14, 5.5), sharex=True)
colores = ["tab:blue", "tab:orange", "tab:green"]
for a, (nombre, toas), col in zip(ax, trenes.items(), colores):
    a.vlines(toas, 0, 1, color=col, lw=2); a.set_yticks([]); a.set_ylabel(nombre, rotation=0, ha="right", va="center")
todos = np.sort(np.concatenate(list(trenes.values())))
ax[3].vlines(todos, 0, 1, color="k", lw=2); ax[3].set_yticks([])
ax[3].set_ylabel("Lo que ve el ESM", rotation=0, ha="right", va="center", fontweight="bold")
ax[3].set_xlabel("Tiempo (µs)")
plt.tight_layout(); plt.show()
print("DTOA de la mezcla:", np.round(np.diff(todos)[:12], 0), "...  ← ¡ninguno es una PRI real!")

### 1.1 Parámetros para separar

| Parámetro | Ventaja | Limitación |
|---|---|---|
| **AOA** | El emisor no puede "saltar" de dirección: es el separador **más robusto** | Precisión de algunos grados; emisores en la misma marcación |
| **RF** | Muy precisa (≈ 1 MHz) | Emisores en la misma frecuencia; **agilidad** |
| **PW** | Fácil de medir | Muchos radares usan PW parecidos; degradación con señal débil |
| **PRI / TOA** | Firma muy específica | **Sólo** se puede medir **después** de separar (o con técnicas especiales: CDIF/SDIF) |

💡 Estrategia clásica: **(1)** separar por parámetros **intrínsecos de cada pulso** (AOA, RF, PW) → **(2)** analizar la PRI y el scan **dentro de cada grupo** → **(3)** unir o dividir grupos según lo que muestre la PRI.

---
## 2. Cargar el escenario

In [ ]:
df = cargar_pdw("D3_ESM_escenario_multiemisor.xlsx")
df.head()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
ax[0].plot(df["TOA_s"], df["AMP_dBm"], ".", ms=1, color="tab:red"); ax[0].set_ylabel("AMP (dBm)")
ax[0].set_title(f"Escenario completo · {len(df):,} pulsos")
ax[1].plot(df["TOA_s"], df["FREQ_MHz"], ".", ms=1); ax[1].set_ylabel("RF (MHz)"); ax[1].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()

### 2.1 ¿Qué pasa si analizamos la PRI de la mezcla?

In [ ]:
graficar_pri(df["TOA_us"], "mezcla completa")
analizar_pri(df["TOA_us"]);

⚠️ **¡Cuidado!** La función automática entrega un resultado **aparentemente razonable** ("Dwell & Switch con 400, 500 y 625 µs")… y es **falso**: 400 µs es la PRI de un emisor y 500/625 µs la de **otro**. Ningún radar del escenario hace Dwell & Switch. En la secuencia (izquierda) se ve que los valores no forman "escalones" largos sino que se mezclan.

🎯 **Lección:** una herramienta automática aplicada a datos mal preparados entrega conclusiones **erróneas con total seguridad**. Sobre una mezcla, la PRI no tiene significado. **Primero hay que separar.**

---
## 3. Explorar el espacio de parámetros

### 3.1 Histogramas de cada parámetro
💡 La RF y el PW abarcan órdenes de magnitud distintos: usamos **escala logarítmica** en el PW.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
ax[0].hist(df["FREQ_MHz"], bins=400); ax[0].set_xlabel("RF (MHz)"); ax[0].set_title("RF")
ax[1].hist(df["AOA_deg"], bins=360); ax[1].set_xlabel("AOA (°)"); ax[1].set_title("AOA")
ax[2].hist(df["PW_us"], bins=np.logspace(-1, 1.5, 200)); ax[2].set_xscale("log"); ax[2].set_xlabel("PW (µs, log)"); ax[2].set_title("PW")
plt.tight_layout(); plt.show()

### ✏️ Tu turno 1
¿Cuántos peaks ves en cada histograma? ¿Coinciden los números? ¿Qué podría explicar que un histograma tenga **menos** peaks que otro?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


RF: **4** peaks (≈ 1300, 3050, 5450 y 9380 MHz). AOA: **5** peaks (≈ 40, 95, 160, 210 y 300°). PW: 4–5 valores (≈ 0,25, 0,6, 1,0, 1,5 y 20 µs).
Un parámetro muestra menos peaks cuando **dos emisores comparten ese valor**: aquí dos emisores comparten la RF ≈ 9380 MHz y se distinguen por AOA y PW. Por eso **ningún parámetro por sí solo basta**.


</details>

### 3.2 El "mapa de emisores": RF vs AOA
📘 En un gráfico de dispersión de dos parámetros cada emisor forma una **nube compacta**. El ruido aparece disperso.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 5.5))
ax[0].scatter(df["AOA_deg"], df["FREQ_MHz"], s=2, alpha=0.4)
ax[0].set_xlabel("AOA (°)"); ax[0].set_ylabel("RF (MHz)"); ax[0].set_title("RF vs AOA")
ax[1].scatter(df["FREQ_MHz"], df["PW_us"], s=2, alpha=0.4)
ax[1].set_yscale("log"); ax[1].set_xlabel("RF (MHz)"); ax[1].set_ylabel("PW (µs, log)"); ax[1].set_title("PW vs RF")
plt.tight_layout(); plt.show()

### 3.3 *Pairplot*: todas las combinaciones de una vez
📘 `sns.pairplot` dibuja una matriz: en la **diagonal** el histograma de cada variable y **fuera** de ella la dispersión de cada par. Usamos una muestra de 3000 pulsos para que sea rápido.

In [ ]:
muestra = df.sample(3000, random_state=1).copy()
muestra["log10_PW"] = np.log10(muestra["PW_us"])
sns.pairplot(muestra[["FREQ_MHz", "AOA_deg", "log10_PW", "AMP_dBm"]], plot_kws=dict(s=4, alpha=0.4),
             diag_kws=dict(bins=80), height=2.4)
plt.show()

💡 La amplitud **no** sirve para separar emisores (depende de la distancia y del scan), pero el pairplot muestra que los pulsos de ruido tienden a estar cerca del umbral (≈ −72 dBm).

---
## 4. Segmentación por banda
📘 `pd.cut` asigna cada valor a un intervalo: perfecto para clasificar la RF en bandas de una sola vez (sin `for`).

In [ ]:
limites_mhz = [0, 1000, 2000, 4000, 8000, 12000, 18000, 27000]
nombres_banda = ["<L", "L", "S", "C", "X", "Ku", "K"]
df["BANDA"] = pd.cut(df["FREQ_MHz"], bins=limites_mhz, labels=nombres_banda)
df["BANDA"].value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(18, 3.5))
for a, b in zip(ax, ["L", "S", "C", "X", "Ku"]):
    sub = df[df["BANDA"] == b]
    a.scatter(sub["AOA_deg"], sub["FREQ_MHz"], s=2, alpha=0.5)
    a.set_title(f"Banda {b} · {len(sub):,} pulsos"); a.set_xlabel("AOA (°)"); a.set_xlim(0, 360)
ax[0].set_ylabel("RF (MHz)")
plt.tight_layout(); plt.show()

🎯 Dentro de cada banda los emisores quedan claramente separados en AOA, salvo el ruido. En **Ku** sólo hay ruido disperso: ningún emisor real.

---
## 5. Separación por ventanas (*gates*)

📘 Una **ventana** es un rectángulo en el espacio de parámetros: "todo pulso con RF entre a y b, AOA entre c y d y PW entre e y f pertenece al grupo G". Se definen **mirando los gráficos** y se aplican con `asignar_por_ventanas`.

### ✏️ Tu turno 2 · Define las ventanas
Completa el diccionario con un grupo por emisor (el primero ya está como ejemplo). Usa márgenes amplios en AOA (±15°): los pulsos débiles tienen más error angular.

In [ ]:
ventanas = {
    "G1": {"FREQ_MHz": (3000, 3100), "AOA_deg": (25, 55)},
    # ✏️ "G2": {"FREQ_MHz": (..., ...), "AOA_deg": (..., ...), "PW_us": (..., ...)},
}
df = asignar_por_ventanas(df, ventanas)
df["GRUPO"].value_counts()

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


```python
ventanas = {
"G1": {"FREQ_MHz": (3000, 3100), "AOA_deg": (25, 55)},
"G2": {"FREQ_MHz": (9350, 9410), "AOA_deg": (80, 110),  "PW_us": (0.4, 0.9)},
"G3": {"FREQ_MHz": (5400, 5500), "AOA_deg": (195, 225)},
"G4": {"FREQ_MHz": (9350, 9410), "AOA_deg": (285, 315), "PW_us": (0.1, 0.4)},
"G5": {"FREQ_MHz": (1250, 1350), "AOA_deg": (140, 180), "PW_us": (12, 30)},
}
```


</details>

▶️ La siguiente celda aplica las ventanas de referencia (para que el resto del cuaderno funcione aunque no hayas terminado el ejercicio). **Compara tus resultados con estos.**

In [ ]:
ventanas = {
    "G1": {"FREQ_MHz": (3000, 3100), "AOA_deg": (25, 55)},
    "G2": {"FREQ_MHz": (9350, 9410), "AOA_deg": (80, 110),  "PW_us": (0.4, 0.9)},
    "G3": {"FREQ_MHz": (5400, 5500), "AOA_deg": (195, 225)},
    "G4": {"FREQ_MHz": (9350, 9410), "AOA_deg": (285, 315), "PW_us": (0.1, 0.4)},
    "G5": {"FREQ_MHz": (1250, 1350), "AOA_deg": (140, 180), "PW_us": (12, 30)},
}
df = asignar_por_ventanas(df, ventanas)
display(df["GRUPO"].value_counts())

plt.figure(figsize=(12, 5.5))
for g, sub in df.groupby("GRUPO"):
    plt.scatter(sub["AOA_deg"], sub["FREQ_MHz"], s=3 if g != "SIN_ASIGNAR" else 2,
                alpha=0.6 if g != "SIN_ASIGNAR" else 0.25, label=g, color="lightgray" if g == "SIN_ASIGNAR" else None)
plt.xlabel("AOA (°)"); plt.ylabel("RF (MHz)"); plt.legend(markerscale=4); plt.title("Pulsos asignados por ventanas")
plt.show()

---
## 6. Caracterizar cada grupo
Ahora sí: dentro de cada grupo hay **un solo emisor** y la PRI vuelve a tener sentido.

In [ ]:
fig, ax = plt.subplots(5, 2, figsize=(16, 16))
for i, g in enumerate(["G1", "G2", "G3", "G4", "G5"]):
    sub = df[df["GRUPO"] == g]
    dt = np.diff(sub["TOA_us"].values); k = min(len(dt), 300)
    ax[i, 0].plot(np.arange(k), dt[:k], ".", ms=3); ax[i, 0].set_ylim(0, np.quantile(dt[:k], 0.9) * 1.6)
    ax[i, 0].set_ylabel(f"{g}\nDTOA (µs)")
    ax[i, 1].plot(sub["TOA_s"], sub["AMP_dBm"], ".", ms=1.5, color="tab:red"); ax[i, 1].set_ylabel("AMP (dBm)")
    ax[i, 1].set_xlim(0, df["TOA_s"].max())
ax[0, 0].set_title("Secuencia de DTOA"); ax[0, 1].set_title("Amplitud vs tiempo (s)")
plt.tight_layout(); plt.show()

In [ ]:
fichas = []
for g in ["G1", "G2", "G3", "G4", "G5"]:
    fichas.append(ficha_emisor(df[df["GRUPO"] == g], g, verbose=False))
lista = pd.DataFrame(fichas).set_index("EMISOR")
columnas = ["N_PULSOS", "T_INICIO_s", "T_FIN_s", "RF_MEDIANA_MHz", "RF_TIPO", "BANDA_IEEE", "BANDA_OTAN",
            "PW_MEDIANA_us", "PRI_TIPO", "PRI_DETALLE", "PRF_MEDIA_Hz", "SCAN_TIPO", "SCAN_DETALLE", "AOA_MEDIA_deg"]
lista[columnas]

### ✏️ Tu turno 3 · Interpreta la lista
1. ¿Qué dos emisores comparten la RF? ¿Qué parámetros permitieron separarlos?
2. ¿Hay algún emisor que **no** estuviera activo durante toda la captura? ¿Desde qué momento aparece?
3. Asigna una **función probable** a cada grupo, con una frase de justificación.

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. **G2 y G4** (≈ 9380 MHz): se separaron por **AOA** (≈ 95° vs ≈ 300°) y **PW** (0,6 vs 0,25 µs). Además tienen PRI y scan distintos.
2. **G4** aparece a los ≈ 24 s (iluminación **continua**, PRF 2,5 kHz, PW corto): posible **seguimiento / lock-on** → evento a reportar con hora.
3. G1: vigilancia aérea banda S (PRI estable 1250 µs, circular 6 s). G2: búsqueda de superficie/adquisición banda X (stagger 500/625, sectorial 2 s). G3: adquisición banda C (jitter ±10 %, circular 4 s). G4: seguimiento/tiro banda X. G5: vigilancia de largo alcance banda L (PRI 3 ms → R<sub>ua</sub> 450 km, PW 20 µs, probablemente con compresión de pulso). Su scan sólo muestra **2 iluminaciones** separadas ≈ 12 s: periodo probable con **baja confianza** (se necesita una captura más larga).


</details>

---
## 7. Línea de tiempo de actividad
🎯 En un informe ESM es fundamental **cuándo** está activo cada emisor (aparición, desaparición, cambios). Graficamos un punto por pulso en la fila de su grupo:

In [ ]:
orden = ["G1", "G2", "G3", "G4", "G5", "SIN_ASIGNAR"]
plt.figure(figsize=(15, 4))
for i, g in enumerate(orden):
    sub = df[df["GRUPO"] == g]
    plt.plot(sub["TOA_s"], np.full(len(sub), i), "|", ms=12, alpha=0.5)
plt.yticks(range(len(orden)), orden); plt.xlabel("Tiempo (s)")
plt.title("Actividad de cada emisor durante la captura"); plt.grid(axis="x", alpha=0.4)
plt.show()

---
## 8. ¿Y los pulsos sin asignar?

In [ ]:
sin = df[df["GRUPO"] == "SIN_ASIGNAR"]
print(f"Pulsos sin asignar: {len(sin)} ({100*len(sin)/len(df):.1f} %) · AMP mediana {sin['AMP_dBm'].median():.1f} dBm")
analizar_pri(sin["TOA_us"]);
graficar_distribucion(sin["FREQ_MHz"], "RF de los pulsos sin asignar", "MHz")

🎯 Están repartidos al azar en RF, AOA y tiempo, cerca del umbral y **sin estructura de PRI**: son **ruido / falsas detecciones**. ⚠️ Antes de descartarlos siempre conviene verificar que no sean un emisor débil o de muy baja PRF (pocos pulsos): busca acumulaciones en el mapa RF–AOA.

---
## 9. Para profundizar: separar **sólo con el tiempo** (idea de CDIF / SDIF)
📘 ¿Y si dos emisores tuvieran la misma RF, el mismo PW y el mismo AOA? Los algoritmos clásicos (**CDIF**, *Cumulative Difference Histogram*, y **SDIF**, *Sequential DIFference*) calculan diferencias de TOA no sólo entre pulsos **consecutivos** (orden 1) sino también **saltando** 1, 2, 3… pulsos (órdenes 2, 3, 4). Las PRI verdaderas aparecen como peaks que **persisten** en varios órdenes.

Probemos con todos los pulsos de banda X a ≈ 9380 MHz (G2 + G4 **mezclados**) durante una iluminación de G2 en que ambos emisores llegan a la vez:

In [ ]:
il2 = detectar_iluminaciones(df[(df["GRUPO"] == "G2") & (df["TOA_s"] > 24.5)])
k = il2["n_pulsos"].idxmax()
t_a, t_b = il2.loc[k, "t_inicio_s"], il2.loc[k, "t_fin_s"]
x = df[df["FREQ_MHz"].between(9350, 9410) & df["TOA_s"].between(t_a, t_b)]
print(f"Ventana {t_a:.3f}–{t_b:.3f} s · {len(x)} pulsos en banda X (sin separar)")
histograma_diferencias(x["TOA_us"], orden_max=4, max_us=1500, bins=300, titulo="banda X, G2 + G4 mezclados")

🎯 **Lectura**
* **Orden 1:** el pico en **400 µs** es la PRI de **G4** (sus pulsos son los más frecuentes); el "peine" de barras bajas son diferencias **cruzadas** entre pulsos de emisores distintos.
* **Órdenes 2 y 3:** aparecen **500 y 625 µs**, la PRI de **G2**: entre dos pulsos consecutivos de G2 casi siempre se cuela uno de G4, por eso su PRI aparece al "saltar" un pulso. 800 y 1200 µs son 2× y 3× la PRI de G4.
* **Orden 4:** **1125 µs = 500 + 625**, el **periodo de trama** del stagger de G2.

Estas técnicas son la base de los procesadores ESM automáticos. Su debilidad: un emisor con muchos pulsos "tapa" a los demás. En el Lab 5 veremos otra estrategia: el **clustering** automático sobre RF/PW/AOA.

---
## 10. Producto: lista de emisores

In [ ]:
os.makedirs("resultados", exist_ok=True)
with pd.ExcelWriter("resultados/D3_lista_emisores.xlsx") as w:
    lista.to_excel(w, sheet_name="EMISORES")
    pd.DataFrame(ventanas).T.astype(str).to_excel(w, sheet_name="VENTANAS")
print("Guardado: resultados/D3_lista_emisores.xlsx ✔")

### Preguntas de repaso
1. ¿Por qué el AOA es el parámetro de separación más robusto? ¿En qué caso falla?
2. ¿Qué pasaría con las ventanas fijas si uno de los emisores tuviera **agilidad en frecuencia** de 300 MHz?
3. Enumera dos limitaciones del método de ventanas manuales en un escenario real con 30 emisores.

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Porque depende de la **geometría** (posición del emisor respecto del receptor), que cambia lentamente, y el emisor no la controla. Falla cuando dos emisores están en la **misma marcación** (p. ej. en el mismo sitio o alineados) o con DF de baja precisión.
2. Una ventana angosta en RF **cortaría** al emisor en varios pedazos o perdería pulsos; habría que usar una ventana ancha en RF y apoyarse en AOA/PW, y luego confirmar con la PRI.
3. (a) Es lento y subjetivo (depende del analista); (b) no escala a muchos emisores; (c) las ventanas rectangulares no se adaptan a nubes inclinadas/alargadas; (d) no entrega una medida de **confianza** de la asignación de cada pulso. → Motivación del **GMM** (Lab 5).


</details>

---
### ✅ Fin del Laboratorio 4

**Antes de continuar**, guarda tu trabajo: *Archivo → Guardar una copia en Drive* (Colab) o *File → Save* (Jupyter).

➡️ **Siguiente:** **Laboratorio 5 · Señales desconocidas con clustering GMM** (separación automática, probabilidades y cotejo con la biblioteca de emisores).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos.</span>